<a href="https://colab.research.google.com/github/canilkr/anil-health-dashboard/blob/main/Sleep_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

# Update this path if your folder name is different
folder_path = '/content/drive/MyDrive/HealthData'

if os.path.exists(folder_path):
    files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    print(f"Found these files in {folder_path}:")
    for f in files:
        print(f"- {f}")
else:
    print("Folder not found. Please check the path in your Drive.")

Folder not found. Please check the path in your Drive.


In [15]:
import json
import math
import pandas as pd
from datetime import datetime, timedelta, date
from IPython.display import display
from google.colab import drive
import os

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Path to your file
file_path = '/content/drive/MyDrive/HealthData/Sleep_Analysis-2026-04-26-2026-05-26.json'

# 3. Load Data
with open(file_path, 'r') as f:
    raw_json = json.load(f)

metrics_list = raw_json.get('data', {}).get('metrics', [])

# 4. Process Sessions
sessions = {}
for item in metrics_list:
    for p in item.get('data', []):
        start_str = p.get('date', '')
        qty = float(p.get('qty', 0))
        if not start_str or qty <= 0: continue

        start_dt = datetime.strptime(start_str[:19], '%Y-%m-%d %H:%M:%S')
        end_dt = start_dt + timedelta(hours=qty)

        # Report date logic: sessions ending after 6 PM are credited to the next day
        report_date = end_dt.strftime('%Y-%m-%d') if end_dt.hour < 18 else (end_dt + timedelta(days=1)).strftime('%Y-%m-%d')

        # Filter: Exclude dates in the future
        if report_date > date.today().strftime('%Y-%m-%d'): continue

        if report_date not in sessions:
            sessions[report_date] = {'deep': [], 'rem': [], 'core': [], 'awake': []}

        val = str(p.get('value', '')).lower()
        interval = (start_dt, end_dt)

        if 'awake' in val: sessions[report_date]['awake'].append(interval)
        elif 'rem' in val: sessions[report_date]['rem'].append(interval)
        elif 'core' in val: sessions[report_date]['core'].append(interval)
        elif 'deep' in val: sessions[report_date]['deep'].append(interval)

# 5. Build Table
sleep_data = []
for d in sorted(sessions.keys(), reverse=True):
    s = sessions[d]

    # Calculate durations in minutes by summing intervals
    m_deep = sum([(i[1]-i[0]).total_seconds() for i in s['deep']]) / 60
    m_rem = sum([(i[1]-i[0]).total_seconds() for i in s['rem']]) / 60
    m_core = sum([(i[1]-i[0]).total_seconds() for i in s['core']]) / 60
    m_awake = sum([(i[1]-i[0]).total_seconds() for i in s['awake']]) / 60

    # Accurate Total: Sum only the categorized stages (removes double-counting)
    total_m = int(m_deep + m_rem + m_core + m_awake)

    # Skip noise data
    if total_m < 60: continue

    sleep_data.append({
        'Date': d,
        'Total Sleep': f"{total_m // 60}h {total_m % 60}m",
        'Deep': f"{int(m_deep)//60}h {int(m_deep)%60}m",
        'REM': f"{int(m_rem)//60}h {int(m_rem)%60}m",
        'Core': f"{int(m_core)//60}h {int(m_core)%60}m",
        'Awake': f"{int(m_awake)//60}h {int(m_awake)%60}m"
    })

# Display table and save to CSV for Lovable
df = pd.DataFrame(sleep_data)
df.to_csv('/content/drive/MyDrive/HealthData/Final_Sleep_Data.csv', index=False)
display(df)

,Date,Total Sleep,Deep,REM,Core,Awake
0,2026-05-26,6h 50m,0h 32m,1h 2m,5h 7m,0h 8m
1,2026-05-25,8h 11m,0h 48m,1h 44m,5h 23m,0h 15m
2,2026-05-24,6h 54m,0h 53m,1h 14m,4h 14m,0h 32m
3,2026-05-23,7h 26m,0h 41m,0h 56m,5h 13m,0h 35m
4,2026-05-22,7h 35m,0h 38m,1h 38m,5h 8m,0h 10m
5,2026-05-21,7h 2m,1h 4m,0h 54m,4h 41m,0h 22m
6,2026-05-20,7h 44m,0h 46m,1h 42m,4h 58m,0h 17m
7,2026-05-19,7h 12m,0h 58m,1h 35m,4h 27m,0h 11m
8,2026-05-18,7h 31m,0h 51m,1h 23m,5h 4m,0h 12m
9,2026-05-17,6h 17m,1h 7m,1h 14m,3h 51m,0h 4m


In [2]:
from google.colab import drive
import os

# 1. Force a clean remount
drive.mount('/content/drive', force_remount=True)

# 2. Define your target folder
folder_path = '/content/drive/MyDrive/HealthData'

# 3. List everything in the folder to see what Python actually sees
print("Files currently visible in 'HealthData' folder:")
if os.path.exists(folder_path):
    print(os.listdir(folder_path))
else:
    print("Folder path not found. Checking root...")
    print(os.listdir('/content/drive/MyDrive/'))

Mounted at /content/drive
Files currently visible in 'HealthData' folder:
['HealthAutoExport-2026-04-24-2026-05-24.json', 'Sleep_Analysis-2026-04-26-2026-05-26.json']


In [6]:
print([item.get('name') for item in metrics_list])

['sleep_analysis']


In [ ]:
import json
import math
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display

# Path to your file
file_path = '/content/drive/MyDrive/HealthData/Sleep_Analysis-2026-04-26-2026-05-26.json'

with open(file_path, 'r') as f:
    raw_json = json.load(f)

metrics_list = raw_json.get('data', {}).get('metrics', [])

# Helper function
def get_duration_mins(intervals):
    if not intervals: return 0
    intervals.sort()
    total_sec = 0
    curr_start, curr_end = intervals[0]
    for next_start, next_end in intervals[1:]:
        if next_start < curr_end: curr_end = max(curr_end, next_end)
        else:
            total_sec += (curr_end - curr_start).total_seconds()
            curr_start, curr_end = next_start, next_end
    total_sec += (curr_end - curr_start).total_seconds()
    return int(math.ceil(total_sec / 60))

# Process Data
sessions = {}
for item in metrics_list:
    for p in item.get('data', []):
        start_dt = datetime.strptime(p.get('date', '')[:19], '%Y-%m-%d %H:%M:%S')
        qty = p.get('qty', 0)
        end_dt = start_dt + timedelta(hours=qty)
        report_date = end_dt.strftime('%Y-%m-%d') if end_dt.hour < 18 else (end_dt + timedelta(days=1)).strftime('%Y-%m-%d')

        if report_date not in sessions:
            sessions[report_date] = {'all_asleep': [], 'deep': [], 'rem': [], 'core': [], 'awake': []}

        interval = (start_dt, end_dt)
        val = str(p.get('value', '')).lower()
        if 'in_bed' in val or 'inbed' in val: continue

        if 'awake' in val: sessions[report_date]['awake'].append(interval)
        elif 'rem' in val: sessions[report_date]['rem'].append(interval)
        elif 'core' in val: sessions[report_date]['core'].append(interval)
        elif 'deep' in val: sessions[report_date]['deep'].append(interval)
        else: sessions[report_date]['all_asleep'].append(interval)

# Build Table
sleep_data = []
for date in sorted(sessions.keys(), reverse=True):
    s = sessions[date]
    total_m = get_duration_mins(s['all_asleep'])
    if total_m < 1: continue

    sleep_data.append({
        'Date': date,
        'Total Sleep': f"{total_m // 60}h {total_m % 60}m",
        'Deep': f"{get_duration_mins(s['deep']) // 60}h {get_duration_mins(s['deep']) % 60}m",
        'REM': f"{get_duration_mins(s['rem']) // 60}h {get_duration_mins(s['rem']) % 60}m",
        'Core': f"{get_duration_mins(s['core']) // 60}h {get_duration_mins(s['core']) % 60}m",
        'Awake': f"{get_duration_mins(s['awake']) // 60}h {get_duration_mins(s['awake']) % 60}m"
    })

display(pd.DataFrame(sleep_data))

In [12]:
# Add this print inside your loop to see the "why"
if report_date > '2026-05-26':
    print(f"Future Date Found: {report_date} | Raw End Time: {end_dt}")

In [13]:
import json
import math
import pandas as pd
from datetime import datetime, timedelta, date
from IPython.display import display
from google.colab import drive
import os

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Path to your file
file_path = '/content/drive/MyDrive/HealthData/Sleep_Analysis-2026-04-26-2026-05-26.json'

# 3. Load Data
with open(file_path, 'r') as f:
    raw_json = json.load(f)

metrics_list = raw_json.get('data', {}).get('metrics', [])

# 4. Process Sessions
sessions = {}
for item in metrics_list:
    for p in item.get('data', []):
        start_str = p.get('date', '')
        qty = float(p.get('qty', 0))
        if not start_str or qty <= 0: continue

        start_dt = datetime.strptime(start_str[:19], '%Y-%m-%d %H:%M:%S')
        end_dt = start_dt + timedelta(hours=qty)

        # Report date logic: sessions ending after 6 PM are credited to the next day
        report_date = end_dt.strftime('%Y-%m-%d') if end_dt.hour < 18 else (end_dt + timedelta(days=1)).strftime('%Y-%m-%d')

        # Filter: Exclude dates in the future
        if report_date > date.today().strftime('%Y-%m-%d'): continue

        if report_date not in sessions:
            sessions[report_date] = {'all': [], 'deep': [], 'rem': [], 'core': [], 'awake': []}

        val = str(p.get('value', '')).lower()
        interval = (start_dt, end_dt)

        if 'awake' in val: sessions[report_date]['awake'].append(interval)
        elif 'rem' in val: sessions[report_date]['rem'].append(interval)
        elif 'core' in val: sessions[report_date]['core'].append(interval)
        elif 'deep' in val: sessions[report_date]['deep'].append(interval)
        else: sessions[report_date]['all'].append(interval)

# 5. Build Table
sleep_data = []
for d in sorted(sessions.keys(), reverse=True):
    s = sessions[d]
    # Sum segments exactly
    m_deep = sum([(i[1]-i[0]).total_seconds() for i in s['deep']]) / 60
    m_rem = sum([(i[1]-i[0]).total_seconds() for i in s['rem']]) / 60
    m_core = sum([(i[1]-i[0]).total_seconds() for i in s['core']]) / 60
    m_awake = sum([(i[1]-i[0]).total_seconds() for i in s['awake']]) / 60
    m_other = sum([(i[1]-i[0]).total_seconds() for i in s['all']]) / 60

    total_m = int(m_deep + m_rem + m_core + m_awake + m_other)
    if total_m < 60: continue

    sleep_data.append({
        'Date': d,
        'Total Sleep': f"{total_m // 60}h {total_m % 60}m",
        'Deep': f"{int(m_deep)//60}h {int(m_deep)%60}m",
        'REM': f"{int(m_rem)//60}h {int(m_rem)%60}m",
        'Core': f"{int(m_core)//60}h {int(m_core)%60}m",
        'Awake': f"{int(m_awake)//60}h {int(m_awake)%60}m"
    })

display(pd.DataFrame(sleep_data))

,Date,Total Sleep,Deep,REM,Core,Awake
0,2026-05-26,417h 49m,0h 32m,1h 2m,5h 7m,0h 8m
1,2026-05-25,383h 45m,0h 48m,1h 44m,5h 23m,0h 15m
2,2026-05-24,225h 0m,0h 53m,1h 14m,4h 14m,0h 32m
3,2026-05-23,248h 17m,0h 41m,0h 56m,5h 13m,0h 35m
4,2026-05-22,587h 1m,0h 38m,1h 38m,5h 8m,0h 10m
5,2026-05-21,184h 19m,1h 4m,0h 54m,4h 41m,0h 22m
6,2026-05-20,318h 4m,0h 46m,1h 42m,4h 58m,0h 17m
7,2026-05-19,284h 32m,0h 58m,1h 35m,4h 27m,0h 11m
8,2026-05-18,231h 39m,0h 51m,1h 23m,5h 4m,0h 12m
9,2026-05-17,307h 29m,1h 7m,1h 14m,3h 51m,0h 4m
